# PRACTICE 3: CLUSTERING RETAIL CUSTOMERS
## TRẢI NGHIỆM SUY LUẬN TƯƠNG TÁC (INTERACTIVE INFERENCE DEMO)

> **Mục tiêu kỹ thuật:**
> - Nạp mô hình quán quân đã lưu trữ `models/saved_models/best_model.joblib` và bộ tiền xử lý `preprocessor.joblib`.
> - Cung cấp hàm dự đoán phân khúc cho khách hàng mới nhập vào trong thời gian thực.
> - Trực quan hóa vị trí khách hàng mới trên không gian 2D so với 5 cụm và các tâm cụm (centroid).
> - Xuất khuyến nghị chiến lược tiếp thị bán lẻ tương ứng với phân khúc được gán.

---
## Mục 1: Khởi Tạo Môi Trường & Nạp Hiện Vật Đã Huấn Luyện

### 1.1 Khai Báo Thư Viện & Nạp Mô Hình Quán Quân
* **Mục tiêu:** Nạp mô hình K-Means Scratch tốt nhất và bộ chuẩn hóa StandardScaler từ thư mục `models/saved_models/`.

In [ ]:
import sys
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.common import SAVED_MODELS_DIR, PROCESSED_CSV, to_rel_path

# Nạp các hiện vật đã huấn luyện
best_model = joblib.load(SAVED_MODELS_DIR / 'best_model.joblib')
preprocessor = joblib.load(SAVED_MODELS_DIR / 'preprocessor.joblib')
with open(SAVED_MODELS_DIR / 'metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

df_clean = pd.read_csv(PROCESSED_CSV)
print(f"✔ Đã nạp thành công mô hình: {metadata['best_model'].upper()} Scratch (K={metadata['optimal_k']})")
print(f"• Đặc trưng sử dụng: {metadata['features_used']}")

---
## Mục 2: Xây Dựng Hàm Dự Đoán Phân Khúc Khách Hàng Mới

### 2.1 Định Nghĩa Hàm Dự Đoán Phân Khúc & Chiến Lược Tiếp Thị
* **Mục tiêu:** Chuẩn hóa thông số đầu vào và trả về tên phân khúc Persona kèm chiến lược bán lẻ tương ứng.

In [ ]:
PERSONA_MAP = {
    0: {
        'name': 'Tiêu xài Tự do (Spenders)',
        'strategy': 'Tung các đợt Flash sales, chương trình khuyến mãi định kỳ, tiếp thị qua mạng xã hội.'
    },
    1: {
        'name': 'Khách hàng VIP (Target)',
        'strategy': 'Chăm sóc bằng chương trình khách hàng thân thiết VIP, dịch vụ cao cấp và đặc quyền độc quyền.'
    },
    2: {
        'name': 'Khách hàng Tiết kiệm (Sensible)',
        'strategy': 'Giới thiệu sản phẩm theo xu hướng giới trẻ, chính sách giá cạnh tranh và combo ưu đãi.'
    },
    3: {
        'name': 'Cẩn trọng / Giữ tiền (Careful)',
        'strategy': 'Quảng bá sản phẩm thương hiệu chất lượng cao, tiếp thị tập trung vào giá trị bền vững và đầu tư lâu dài.'
    },
    4: {
        'name': 'Khách hàng Tiêu chuẩn (Standard)',
        'strategy': 'Cung cấp các mặt hàng thiết yếu thường ngày, chương trình tích điểm giảm giá.'
    }
}


def predict_customer(income_k: float, spending_score: float, visualize: bool = True):
    # 1. Chuẩn hóa đặc trưng đầu vào qua StandardScaler
    X_new = np.array([[income_k, spending_score]])
    X_new_scaled = preprocessor.transform(X_new)

    # 2. Dự đoán nhãn phân cụm qua hàm predict
    cluster_id = int(best_model.predict(X_new_scaled)[0])
    persona = PERSONA_MAP[cluster_id]

    print("=" * 70)
    print("   🎯 KẾT QUẢ PHÂN KHÚC KHÁCH HÀNG MỚI (INFERENCE RESULT)")
    print("=" * 70)
    print(f" • Thu nhập hàng năm : ${income_k:,.1f}k / năm")
    print(f" • Điểm chi tiêu     : {spending_score} / 100")
    print(f" • Phân cụm gán      : Cụm {cluster_id}")
    print(f" • Persona Khách hàng: {persona['name']}")
    print(f" • Chiến lược Bán lẻ : {persona['strategy']}")
    print("=" * 70)

    if visualize:
        plt.figure(figsize=(10, 6))
        colors = ['#e53e3e', '#3182ce', '#38a169', '#d69e2e', '#805ad5']
        
        # Vẽ các điểm nền
        for c in range(5):
            mask = best_model.labels_ == c
            plt.scatter(df_clean.loc[mask, 'Annual_Income_k$'], df_clean.loc[mask, 'Spending_Score'],
                        color=colors[c], s=40, alpha=0.4, edgecolors='none', label=PERSONA_MAP[c]['name'])

        # Vẽ khách hàng mới bằng ngôi sao vàng nổi bật
        plt.scatter(income_k, spending_score, color='#ecc94b', marker='*', s=350,
                    edgecolors='black', linewidth=1.5, label='Khách hàng mới nhập vào')

        # Vẽ các tâm cụm Centroids
        centers = preprocessor.inverse_transform(best_model.cluster_centers_)
        plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=150, edgecolors='white', linewidth=1.5, label='Centroids')

        plt.title(f"Vị trí Khách hàng mới trên Không gian Phân cụm 2D (Cụm {cluster_id}: {persona['name']})", fontsize=12, fontweight='bold')
        plt.xlabel("Thu nhập hàng năm (Annual Income k$)")
        plt.ylabel("Điểm chi tiêu (Spending Score 1-100)")
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
        plt.tight_layout()
        plt.show()

    return cluster_id, persona

---
## Mục 3: Thử Nghiệm Suy Luận Với Các Khách Hàng Thực Tế

### 3.1 Thử Nghiệm 1: Khách Hàng Thu Nhập Cao & Chi Tiêu Mạnh (VIP)
* **Thông số:** Thu nhập $90.0k / năm, Điểm chi tiêu 85 / 100.

In [ ]:
res1 = predict_customer(income_k=90.0, spending_score=85, visualize=True)

### 3.2 Thử Nghiệm 2: Khách Hàng Thu Nhập Thấp Nhưng Chi Tiêu Mạnh (Sensible)
* **Thông số:** Thu nhập $25.0k / năm, Điểm chi tiêu 80 / 100.

In [ ]:
res2 = predict_customer(income_k=25.0, spending_score=80, visualize=False)

### 3.3 Thử Nghiệm 3: Khách Hàng Thu Nhập Cao Nhưng Cẩn Trọng Chi Tiêu (Careful)
* **Thông số:** Thu nhập $88.0k / năm, Điểm chi tiêu 15 / 100.

In [ ]:
res3 = predict_customer(income_k=88.0, spending_score=15, visualize=False)